# SRASTA — Training TCN-Lite CSI-Bench

Notebook ini versi `.ipynb` dari `train_tcn_lite_csi.py`.

Fungsi:
- baca dataset CSI-Bench / dataset lokal `.h5`
- preprocessing window CSI
- training model TCN-Lite
- export `model.keras`
- export `model_int8.tflite`
- simpan `label_map.json` dan `training_summary.json`


## 1. Install dependency

In [ ]:
%pip install -q tensorflow numpy scipy h5py scikit-learn pandas requests

## 2. Load fungsi training

Cell ini memuat isi `train_tcn_lite_csi.py`, tapi bagian `main()` tidak dijalankan otomatis.


In [ ]:
#!/usr/bin/env python3
"""
train_tcn_lite_csi.py

Training TCN-Lite untuk data WiFi CSI:
- Membaca file .h5 dari CSI-Bench atau data prototipe.
- Label diambil dari folder "act_<ActivityName>" bila metadata tidak dipakai.
- Mengelompokkan label ke kelas SRASTA: no_person, normal, inactive, fall.
- Melatih model TCN-Lite ringan.
- Mengekspor model Keras dan TensorFlow Lite INT8.

Contoh:
python train_tcn_lite_csi.py \
  --data-root /path/to/CSI-Bench \
  --task FallDetection \
  --out runs/srasta_fall \
  --win-len 500 \
  --feature-size 232 \
  --epochs 50
"""

import argparse
import json
import os
import random
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import h5py
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight


SRASTA_CLASSES = ["no_person", "normal", "inactive", "fall"]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def locate_task_dir(data_root: Path, task: str) -> Path:
    candidates = [
        data_root / task,
        data_root / "tasks" / task,
        data_root / "CSI-Bench" / task,
        data_root / "CSI-Bench" / "tasks" / task,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Task directory tidak ditemukan untuk task={task}. "
        f"Coba pastikan data-root menunjuk ke folder CSI-Bench."
    )


def find_h5_files(task_dir: Path) -> List[Path]:
    files = sorted(task_dir.rglob("*.h5")) + sorted(task_dir.rglob("*.hdf5"))
    if not files:
        raise FileNotFoundError(f"Tidak ada file .h5/.hdf5 di {task_dir}")
    return files


def label_from_path(path: Path) -> str:
    for part in path.parts:
        if part.startswith("act_"):
            return part.replace("act_", "").strip()
    # fallback: parent folder
    return path.parent.name.replace("act_", "").strip()


def to_srasta_label(raw_label: str) -> str:
    """
    Sesuaikan mapping ini dengan label asli dataset/prototipe kalian.
    Untuk lomba, lebih rapi bila label_map akhir tetap 4 kelas:
    no_person, normal, inactive, fall.
    """
    s = raw_label.lower().replace("-", "_").replace(" ", "_")

    if any(k in s for k in ["fall", "fallen", "drop"]):
        return "fall"

    if any(k in s for k in ["empty", "vacant", "no_person", "nobody", "none"]):
        return "no_person"

    # diam/tidur/berbaring dipakai sebagai kelas inactive.
    # Kalau data tidak punya kelas diam, state inactive tetap bisa dibuat dari durasi
    # dan energy CSI pada inference.
    if any(k in s for k in ["inactive", "still", "sleep", "lie", "lying", "sit_still", "stand_still"]):
        return "inactive"

    return "normal"


def h5_numeric_datasets(h5: h5py.File) -> List[Tuple[str, h5py.Dataset]]:
    out = []

    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            if np.issubdtype(obj.dtype, np.number) or np.issubdtype(obj.dtype, np.complexfloating):
                if obj.ndim >= 1 and np.prod(obj.shape) > 0:
                    out.append((name, obj))

    h5.visititems(visitor)
    return out


def choose_csi_dataset(h5: h5py.File) -> Tuple[str, h5py.Dataset]:
    datasets = h5_numeric_datasets(h5)
    if not datasets:
        raise ValueError("Tidak menemukan numeric dataset di file HDF5.")

    def score(item):
        name, ds = item
        n = int(np.prod(ds.shape))
        bonus = 1000000 if any(k in name.lower() for k in ["csi", "channel", "data"]) else 0
        return bonus + n

    return max(datasets, key=score)


def load_csi_h5(path: Path, max_time: int = 0) -> np.ndarray:
    with h5py.File(path, "r") as h5:
        name, ds = choose_csi_dataset(h5)
        arr = np.asarray(ds)

    # Banyak data CSI tersimpan complex. Untuk deployment ringan, pakai amplitude.
    if np.iscomplexobj(arr):
        arr = np.abs(arr)
    else:
        arr = arr.astype(np.float32, copy=False)

    # Asumsi axis waktu adalah axis pertama. Jika data 1D, ubah menjadi [T, 1].
    if arr.ndim == 1:
        arr = arr[:, None]
    elif arr.ndim >= 2:
        arr = arr.reshape(arr.shape[0], -1)

    if max_time and arr.shape[0] > max_time:
        arr = arr[:max_time]

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return arr


def resize_features(x: np.ndarray, feature_size: int) -> np.ndarray:
    """Pad atau subsample feature agar bentuknya [T, feature_size]."""
    t, f = x.shape
    if f == feature_size:
        return x
    if f > feature_size:
        idx = np.linspace(0, f - 1, feature_size).astype(np.int64)
        return x[:, idx]
    pad = np.zeros((t, feature_size - f), dtype=x.dtype)
    return np.concatenate([x, pad], axis=1)


def normalize_window(w: np.ndarray) -> np.ndarray:
    # Normalisasi per feature dalam satu window, cocok untuk variasi ruangan.
    mean = w.mean(axis=0, keepdims=True)
    std = w.std(axis=0, keepdims=True) + 1e-6
    return (w - mean) / std


def make_windows(
    arr: np.ndarray,
    label_id: int,
    win_len: int,
    stride: int,
    feature_size: int,
    max_windows_per_file: int,
) -> Tuple[List[np.ndarray], List[int]]:
    arr = resize_features(arr, feature_size)
    xs, ys = [], []

    if arr.shape[0] < win_len:
        padded = np.zeros((win_len, feature_size), dtype=np.float32)
        padded[: arr.shape[0]] = arr
        xs.append(normalize_window(padded))
        ys.append(label_id)
        return xs, ys

    starts = list(range(0, arr.shape[0] - win_len + 1, stride))
    if max_windows_per_file and len(starts) > max_windows_per_file:
        starts = random.sample(starts, max_windows_per_file)
        starts.sort()

    for st in starts:
        w = arr[st : st + win_len]
        xs.append(normalize_window(w).astype(np.float32))
        ys.append(label_id)

    return xs, ys


def build_tcn_lite(win_len: int, feature_size: int, num_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(win_len, feature_size), name="csi_window")
    x = tf.keras.layers.Conv1D(32, 1, padding="same")(inp)

    for dilation in [1, 2, 4, 8]:
        res = x
        x = tf.keras.layers.SeparableConv1D(
            32,
            kernel_size=5,
            dilation_rate=dilation,
            padding="same",
            activation=None,
        )(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.Dropout(0.10)(x)

        x = tf.keras.layers.SeparableConv1D(
            32,
            kernel_size=3,
            dilation_rate=dilation,
            padding="same",
            activation=None,
        )(x)
        x = tf.keras.layers.BatchNormalization()(x)

        if res.shape[-1] != x.shape[-1]:
            res = tf.keras.layers.Conv1D(32, 1, padding="same")(res)

        x = tf.keras.layers.Add()([x, res])
        x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.15)(x)
    out = tf.keras.layers.Dense(num_classes, activation="softmax", name="class_prob")(x)

    model = tf.keras.Model(inp, out, name="SRASTA_TCN_Lite")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def representative_dataset(x_train: np.ndarray, count: int = 100):
    n = min(count, len(x_train))
    for i in range(n):
        yield [x_train[i : i + 1].astype(np.float32)]


def export_tflite_int8(model: tf.keras.Model, x_train: np.ndarray, out_path: Path) -> None:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = lambda: representative_dataset(x_train, 100)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    try:
        tflite_model = converter.convert()
    except Exception as exc:
        print("[WARN] INT8 full-integer gagal, fallback ke dynamic range:", exc)
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite_model = converter.convert()

    out_path.write_bytes(tflite_model)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", required=True, help="Root dataset CSI-Bench")
    ap.add_argument("--task", default="FallDetection", help="Nama task: FallDetection / HumanActivityRecognition")
    ap.add_argument("--out", default="runs/srasta_tcn_lite")
    ap.add_argument("--win-len", type=int, default=500)
    ap.add_argument("--stride", type=int, default=250)
    ap.add_argument("--feature-size", type=int, default=232)
    ap.add_argument("--epochs", type=int, default=50)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--max-files", type=int, default=0)
    ap.add_argument("--max-windows-per-file", type=int, default=20)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    set_seed(args.seed)

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    task_dir = locate_task_dir(Path(args.data_root), args.task)
    files = find_h5_files(task_dir)
    if args.max_files:
        files = files[: args.max_files]

    label_to_id: Dict[str, int] = {name: i for i, name in enumerate(SRASTA_CLASSES)}
    xs, ys, skipped = [], [], []

    print(f"[INFO] Membaca {len(files)} file dari {task_dir}")
    for i, f in enumerate(files, 1):
        raw_label = label_from_path(f)
        srasta_label = to_srasta_label(raw_label)
        label_id = label_to_id[srasta_label]

        try:
            arr = load_csi_h5(f)
            xw, yw = make_windows(
                arr,
                label_id=label_id,
                win_len=args.win_len,
                stride=args.stride,
                feature_size=args.feature_size,
                max_windows_per_file=args.max_windows_per_file,
            )
            xs.extend(xw)
            ys.extend(yw)
        except Exception as exc:
            skipped.append({"file": str(f), "error": str(exc)})

        if i % 50 == 0:
            print(f"[INFO] processed {i}/{len(files)} files, windows={len(xs)}, skipped={len(skipped)}")

    if len(xs) < 10:
        raise RuntimeError(
            "Window training terlalu sedikit. Cek struktur dataset, task, dan key HDF5."
        )

    X = np.stack(xs).astype(np.float32)
    y = np.asarray(ys, dtype=np.int64)

    # Buang kelas yang tidak muncul agar training tidak error.
    present = sorted(np.unique(y).tolist())
    old_to_new = {old: new for new, old in enumerate(present)}
    new_classes = [SRASTA_CLASSES[old] for old in present]
    y = np.asarray([old_to_new[int(v)] for v in y], dtype=np.int64)
    label_map = {name: i for i, name in enumerate(new_classes)}

    (x_train, x_tmp, y_train, y_tmp) = train_test_split(
        X, y, test_size=0.30, random_state=args.seed, stratify=y if len(np.unique(y)) > 1 else None
    )
    (x_val, x_test, y_val, y_test) = train_test_split(
        x_tmp, y_tmp, test_size=0.50, random_state=args.seed, stratify=y_tmp if len(np.unique(y_tmp)) > 1 else None
    )

    model = build_tcn_lite(args.win_len, args.feature_size, len(new_classes))
    model.summary()

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train,
    )
    class_weight_dict = {int(i): float(w) for i, w in zip(np.unique(y_train), class_weights)}

    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_accuracy"),
        tf.keras.callbacks.ModelCheckpoint(out_dir / "model.keras", save_best_only=True, monitor="val_accuracy"),
    ]

    hist = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=args.epochs,
        batch_size=args.batch_size,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1,
    )

    model = tf.keras.models.load_model(out_dir / "model.keras")

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    prob = model.predict(x_test, verbose=0)
    pred = prob.argmax(axis=1)

    report = classification_report(y_test, pred, target_names=new_classes, output_dict=True)
    cm = confusion_matrix(y_test, pred).tolist()

    (out_dir / "label_map.json").write_text(json.dumps(label_map, indent=2), encoding="utf-8")
    (out_dir / "skipped_files.json").write_text(json.dumps(skipped, indent=2), encoding="utf-8")

    summary = {
        "task": args.task,
        "win_len": args.win_len,
        "feature_size": args.feature_size,
        "num_windows": int(len(X)),
        "classes": new_classes,
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "classification_report": report,
        "confusion_matrix": cm,
        "history": {k: [float(v) for v in vals] for k, vals in hist.history.items()},
        "notes": "Training awal perlu divalidasi ulang dengan data prototipe ESP32-S3 di ruangan target.",
    }
    (out_dir / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    export_tflite_int8(model, x_train, out_dir / "model_int8.tflite")

    print("\n[DONE]")
    print(f"Keras model : {out_dir / 'model.keras'}")
    print(f"TFLite INT8 : {out_dir / 'model_int8.tflite'}")
    print(f"Label map   : {out_dir / 'label_map.json'}")
    print(f"Test acc    : {test_acc:.4f}")


## 3. Konfigurasi path dataset

Ubah `DATA_ROOT` sesuai lokasi dataset.

Contoh:
- CSI-Bench: `data/csi_bench`
- dataset lokal: `data/local_srasta`


In [ ]:
from pathlib import Path
import argparse

DATA_ROOT = "data/csi_bench"      # ubah sesuai lokasi dataset
TASK = "FallDetection"
OUT_DIR = "runs/srasta_fall"

WIN_LEN = 500
STRIDE = 250
FEATURE_SIZE = 232
EPOCHS = 50
BATCH_SIZE = 32

# Isi 100/200 untuk tes cepat. Isi 0 untuk pakai semua file.
MAX_FILES = 0
MAX_WINDOWS_PER_FILE = 20

args = argparse.Namespace(
    data_root=DATA_ROOT,
    task=TASK,
    out=OUT_DIR,
    win_len=WIN_LEN,
    stride=STRIDE,
    feature_size=FEATURE_SIZE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    max_files=MAX_FILES,
    max_windows_per_file=MAX_WINDOWS_PER_FILE,
    seed=42,
)

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
args

## 4. Jalankan training

Kalau baru coba pertama kali, set `MAX_FILES = 100` dan `EPOCHS = 5` dulu agar cepat.


In [ ]:
set_seed(args.seed)

out_dir = Path(args.out)
out_dir.mkdir(parents=True, exist_ok=True)

task_dir = locate_task_dir(Path(args.data_root), args.task)
files = find_h5_files(task_dir)
if args.max_files:
    files = files[: args.max_files]

label_to_id = {name: i for i, name in enumerate(SRASTA_CLASSES)}
xs, ys, skipped = [], [], []

print(f"[INFO] Membaca {len(files)} file dari {task_dir}")
for i, f in enumerate(files, 1):
    raw_label = label_from_path(f)
    srasta_label = to_srasta_label(raw_label)
    label_id = label_to_id[srasta_label]

    try:
        arr = load_csi_h5(f)
        xw, yw = make_windows(
            arr,
            label_id=label_id,
            win_len=args.win_len,
            stride=args.stride,
            feature_size=args.feature_size,
            max_windows_per_file=args.max_windows_per_file,
        )
        xs.extend(xw)
        ys.extend(yw)
    except Exception as exc:
        skipped.append({"file": str(f), "error": str(exc)})

    if i % 50 == 0:
        print(f"[INFO] processed {i}/{len(files)} files, windows={len(xs)}, skipped={len(skipped)}")

if len(xs) < 10:
    raise RuntimeError("Window training terlalu sedikit. Cek struktur dataset, task, dan key HDF5.")

X = np.stack(xs).astype(np.float32)
y = np.asarray(ys, dtype=np.int64)

present = sorted(np.unique(y).tolist())
old_to_new = {old: new for new, old in enumerate(present)}
new_classes = [SRASTA_CLASSES[old] for old in present]
y = np.asarray([old_to_new[int(v)] for v in y], dtype=np.int64)
label_map = {name: i for i, name in enumerate(new_classes)}

(x_train, x_tmp, y_train, y_tmp) = train_test_split(
    X, y, test_size=0.30, random_state=args.seed, stratify=y if len(np.unique(y)) > 1 else None
)
(x_val, x_test, y_val, y_test) = train_test_split(
    x_tmp, y_tmp, test_size=0.50, random_state=args.seed, stratify=y_tmp if len(np.unique(y_tmp)) > 1 else None
)

model = build_tcn_lite(args.win_len, args.feature_size, len(new_classes))
model.summary()

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train,
)
class_weight_dict = {int(i): float(w) for i, w in zip(np.unique(y_train), class_weights)}

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ModelCheckpoint(out_dir / "model.keras", save_best_only=True, monitor="val_accuracy"),
]

hist = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=args.epochs,
    batch_size=args.batch_size,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

model = tf.keras.models.load_model(out_dir / "model.keras")

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
prob = model.predict(x_test, verbose=0)
pred = prob.argmax(axis=1)

report = classification_report(y_test, pred, target_names=new_classes, output_dict=True)
cm = confusion_matrix(y_test, pred).tolist()

(out_dir / "label_map.json").write_text(json.dumps(label_map, indent=2), encoding="utf-8")
(out_dir / "skipped_files.json").write_text(json.dumps(skipped, indent=2), encoding="utf-8")

summary = {
    "task": args.task,
    "win_len": args.win_len,
    "feature_size": args.feature_size,
    "num_windows": int(len(X)),
    "classes": new_classes,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "classification_report": report,
    "confusion_matrix": cm,
    "history": {k: [float(v) for v in vals] for k, vals in hist.history.items()},
    "notes": "Training awal perlu divalidasi ulang dengan data prototipe ESP32-S3 di ruangan target.",
}
(out_dir / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

export_tflite_int8(model, x_train, out_dir / "model_int8.tflite")

print("\n[DONE]")
print(f"Keras model : {out_dir / 'model.keras'}")
print(f"TFLite INT8 : {out_dir / 'model_int8.tflite'}")
print(f"Label map   : {out_dir / 'label_map.json'}")
print(f"Test acc    : {test_acc:.4f}")

## 5. Cek ringkasan hasil training

In [ ]:
import json
from pathlib import Path
import numpy as np

summary_path = Path(OUT_DIR) / "training_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("Classes:", summary["classes"])
    print("Num windows:", summary["num_windows"])
    print("Test accuracy:", summary["test_accuracy"])
    print("Confusion matrix:")
    print(np.array(summary["confusion_matrix"]))
else:
    print("training_summary.json belum ada. Jalankan training dulu.")